In [ ]:
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(8)

!ollama pull llama3.1:8b-instruct-q4_K_M
print("Ollama ready.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy/main 

In [ ]:
!pip install -q requests

In [ ]:
import zipfile
import json
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import os
import time
import random
import requests
from google.colab import userdata
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import re

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
#checking whether ollama is running
resp = requests.get("http://localhost:11434/api/tags")
print("Ollama status:", resp.status_code)
models = [m["name"] for m in resp.json().get("models", [])]
print("Available models:", models)

Ollama status: 200
Available models: ['llama3.1:8b-instruct-q4_K_M']


In [ ]:
OLLAMA_MODEL = "llama3.1:8b-instruct-q4_K_M"
OLLAMA_URL = "http://localhost:11434/api/chat"

In [ ]:
LABEL_DESC = "aapd_label_descriptions_prompt_friendly.json"
VAL_GREEDY_INPUT = "aapd_validation_greedy_llm_input_data.jsonl"
VAL_TOPK_INPUT = "aapd_validation_top_k_llm_input_data.jsonl"
VAL_GOLD_LABELS = "aapd_validation_gold_labels.json"
N_RETRIEVED = 5

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/thesis_results/AAPD_LLAMA_OLLAMA_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GREEDY_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_greedy_predictions.jsonl"
TOPK_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_topk_predictions.jsonl"
LABELS_ONLY_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_labels_only_predictions.jsonl"

Mounted at /content/drive


In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

print("Validation shape:", aapd_df_val.shape)
print("Test shape:", aapd_df_test.shape)
#ok

Validation shape: (1000, 3)
Test shape: (1000, 3)


In [ ]:
#loading retrieval results

#helper function
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

#validation only for now
val_greedy_items = load_jsonl(VAL_GREEDY_INPUT)
val_topk_items = load_jsonl(VAL_TOPK_INPUT)

print(val_greedy_items[0].keys())
print(val_greedy_items[0])

dict_keys(['query_id', 'target_text', 'retrieved_examples'])
{'query_id': 0, 'target_text': 'linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge was to investigate novel approaches to obtain such semantic data in particular , we were seeking methods and tools to extract information from scholarly publications , to publish it as lod , and to use queries over this lod to assess quality this year we focused on the quality of workshop proceedings , and of journal articles w r t their citation network a third , open task , asked to showcase how such semantic data could be exploited and how semantic web technologies could help in this emerging context\n', 'retrieved_examples': [{'train_index': 26088, 'labels': ['Databases', 'Computational Engineering, Finance, and Science', 'Information Retrieval'], 'text': 'the linked clinical trials \\( linkedc

In [ ]:
#for evaluating the validation results
with open(VAL_GOLD_LABELS, "r", encoding="utf-8") as f:
    val_gold_labels = json.load(f)

In [ ]:
#reusing the same mlb I created for SVC
mlb = joblib.load("mlb.joblib")
ALL_LABELS = list(mlb.classes_)
print("Number of labels:", len(ALL_LABELS))

#label block for inserting into prompt
with open(LABEL_DESC, "r", encoding="utf-8") as f:
    label_descriptions = json.load(f)

#check that descriptions and mlb labels match
desc_labels = set(label_descriptions.keys())
mlb_labels = set(ALL_LABELS)
assert mlb_labels == desc_labels, "Mismatch!"

Number of labels: 54


In [ ]:
OLLAMA_OPTIONS = {
    "temperature": 0,
    "num_predict": 192,
    "seed": SEED}

In [ ]:
def label_block(label_descriptions, labels):
    lines = []
    for label in labels:
        desc = label_descriptions[label]
        lines.append(f" - {label}: {desc}")
    return "\n".join(lines)

LABEL_BLOCK = label_block(label_descriptions, ALL_LABELS)

print(LABEL_BLOCK)
#will be provided in-context
print("Number of labels:", len(ALL_LABELS)) #ok
print(f"Number of words:{len(LABEL_BLOCK.split())}")

 - Adaptation and Self-Organizing Systems: Focuses on systems that adapt and self-organize, including statistical physics and stochastic processes. Does not cover general machine learning topics outside of self-organization. Relevant to physics applications in complex systems.
 - Applications: Covers the application of statistical methods across various fields such as biology, engineering, and social sciences. Does not include theoretical statistics or methodology. Emphasizes practical implementations and case studies.
 - Artificial Intelligence: Encompasses all AI topics except for Vision, Robotics, Machine Learning, Multiagent Systems, and Computation and Language. Does not include practical applications of AI methods, which may fall under Applications. Focuses on theoretical aspects like knowledge representation and planning.
 - Combinatorics: Covers discrete mathematics topics such as graph theory, enumeration, and combinatorial optimization. Does not include continuous mathematics

In [ ]:
label_schema = {
    "type": "object",
    "properties": {
        "predicted_labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ALL_LABELS},
        }
    },
    "required": ["predicted_labels"],
    "additionalProperties": False}


In [ ]:
def warmup_ollama():
    """Send a short request to load the model into GPU memory."""
    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": "Hi"}],
        "stream": False,
        "options": {"num_predict": 5}
    })
    print("Ollama warmup done:", resp.status_code)

In [ ]:
warmup_ollama()

Ollama warmup done: 200


In [ ]:
#check
print("Validation df length:", len(aapd_df_val))
print("Greedy items length:", len(val_greedy_items))
print("Top-k items length:", len(val_topk_items))
#checking whether the first target text is the same as val
print(val_greedy_items[0]["target_text"][:200])
print(aapd_df_val.iloc[0]["text"][:200])


Validation df length: 1000
Greedy items length: 1000
Top-k items length: 1000
linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge 
linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge 


In [ ]:
#since the AAPD abstracts are preprocessed in a way that punctuation appears after spaces (making them separate tokens),
#which is useful for SVC/transformers but not necessarily for LLMs
#I will send the text as more natural looking
def detokenize(text):
    text = str(text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\s+", " ", text)
    return text

#formatting the retrieved examples for in-prompt placement
def format_retrieved_examples(retrieved_examples, n=N_RETRIEVED):
    example_blocks = []
    for i, ex in enumerate(retrieved_examples[:n], start=1):
        text = detokenize(ex["text"])
        labels = ex["labels"]

        example_blocks.append(
            f"Example {i}\n"
            f"Abstract:\n{text}\n"
            f"Labels:\n{json.dumps(labels, ensure_ascii=False)}")
    return "\n\n".join(example_blocks)


##Building prompts

In [ ]:
def build_prompt_with_examples(target_text, retrieved_examples):
    target_text = detokenize(target_text)
    examples_block = format_retrieved_examples(retrieved_examples)

    system_message = (
        "You are an expert in multi-label topic classification. "
        "Use only the allowed label names and return only the required JSON object.")

    user_message = f"""
You are performing multi-label topic classification for academic abstracts.
Your task is to assign the most appropriate topic labels to the target abstract.
Use only labels from the allowed label set below.

Allowed labels and descriptions:
{LABEL_BLOCK}

Below are retrieved labelled demonstrations. They show examples of abstracts and their assigned labels.
{examples_block}

Now classify the target abstract.

Target abstract:
{target_text}

Return exactly one JSON object in this format:
{{"predicted_labels": ["label1", "label2"]}}

Rules:
- Use only exact label names from the allowed label set.
- Select only labels that are directly supported by the main topic of the target abstract.
- Use the retrieved demonstrations as guidance, but do not copy their labels unless the target abstract itself supports them.
- Return only the JSON object, with no explanation or extra text.

""".strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}]

In [ ]:
def build_prompt_labels_only(target_text):
    target_text = detokenize(target_text)

    system_message = (
        "You are an expert in multi-label topic classification. "
        "Use only the allowed label names and return only the required JSON object.")

    user_message = f"""
You are performing multi-label topic classification for academic abstracts.
Your task is to assign the most appropriate topic labels to the target abstract.
Use only labels from the allowed label set below.

Allowed labels and descriptions:
{LABEL_BLOCK}

Now classify the target abstract.

Target abstract:
{target_text}

Return exactly one JSON object in this format:
{{"predicted_labels": ["label1", "label2"]}}

Rules:
- Use only exact label names from the allowed label set.
- Select only labels that are directly supported by the main topic of the target abstract.
- Return only the JSON object, with no explanation or extra text.

""".strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}]

In [ ]:
def build_greedy_messages(item):
    return build_prompt_with_examples(target_text=item["target_text"], retrieved_examples=item["retrieved_examples"])


def build_topk_messages(item):
    return build_prompt_with_examples(target_text=item["target_text"], retrieved_examples=item["retrieved_examples"])


def build_labels_only_messages(item):
    return build_prompt_labels_only(target_text=item["target_text"])

In [ ]:
def generate_ollama_response(messages):
    total_start = time.time()

    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": messages,
        "format": label_schema,
        "stream": False,
        "options": OLLAMA_OPTIONS
    })
    resp.raise_for_status()
    data = resp.json()

    total_runtime = time.time() - total_start
    raw_output = data["message"]["content"].strip()

    # Ollama returns token counts and timing natively
    input_tokens = data.get("prompt_eval_count")
    output_tokens = data.get("eval_count")
    # eval_duration is in nanoseconds
    eval_ns = data.get("eval_duration", 0)
    generation_runtime = eval_ns / 1e9

    return {
        "response": raw_output,
        "generation_runtime_seconds": generation_runtime,
        "total_sample_runtime_seconds": total_runtime,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

In [ ]:
def parse_schema_output(raw_output):
    raw_output = str(raw_output).strip()
    parsed = json.loads(raw_output)

    labels = parsed["predicted_labels"]

    if not isinstance(labels, list):
        raise ValueError(f"predicted_labels is not a list: {raw_output}")

    invalid_labels = [
        label for label in labels
        if label not in ALL_LABELS
    ]

    if invalid_labels:
        raise ValueError(f"Invalid labels: {invalid_labels}")

    valid_labels = []
    for label in labels:
        if label not in valid_labels:
            valid_labels.append(label)

    return {
        "predicted_labels": valid_labels,
        "raw_output": raw_output}

In [ ]:
def get_done_query_ids(output_path):
    done = set()

    if not os.path.exists(output_path):
        return done

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    row = json.loads(line)
                    if row.get("status") == "ok":
                        done.add(row["query_id"])
                except Exception:
                    pass
    return done

In [ ]:
def run_ollama_validation_checkpointed(
    items,
    output_path,
    strategy_name,
    build_messages_fn,
    gold_labels_dict,
    max_items=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if max_items is not None:
        items = items[:max_items]

    done_query_ids = get_done_query_ids(output_path)

    remaining_items = [
        item for item in items
        if item["query_id"] not in done_query_ids]

    print(f"Already completed: {len(done_query_ids)}")
    print(f"Remaining to run: {len(remaining_items)}")

    experiment_start = time.time()

    for item in remaining_items:
        query_id = item["query_id"]
        sample_start = time.time()

        try:
            messages = build_messages_fn(item)

            generation = generate_ollama_response(messages)
            parsed = parse_schema_output(generation["response"])

            row = {
                "status": "ok",
                "strategy": strategy_name,
                "query_id": query_id,
                "gold_labels": gold_labels_dict[str(query_id)],
                "predicted_labels": parsed["predicted_labels"],
                "raw_output": parsed["raw_output"],
                "input_tokens": generation["input_tokens"],
                "output_tokens": generation["output_tokens"],
                "generation_runtime_seconds": generation["generation_runtime_seconds"],
                "total_sample_runtime_seconds": generation["total_sample_runtime_seconds"],
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        except Exception as e:
            row = {
                "status": "error",
                "strategy": strategy_name,
                "query_id": query_id,
                "error": repr(e),
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"Finished. Total wall time: {(time.time() - experiment_start) / 60:.2f} minutes")

In [ ]:
def load_prediction_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                if row.get("status") == "ok":
                    rows.append(row)

    return rows

def load_prediction_jsonl_all(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


def evaluate_prediction_file(path, mlb=mlb, require_no_errors=True):
    all_rows = load_prediction_jsonl_all(path)

    ok_rows = [row for row in all_rows if row.get("status") == "ok"]
    error_rows = [row for row in all_rows if row.get("status") == "error"]

    if require_no_errors and len(error_rows) > 0:
        raise ValueError(
            f"{path} contains {len(error_rows)} error rows. ")

    y_true_labels = [row["gold_labels"] for row in ok_rows]
    y_pred_labels = [row["predicted_labels"] for row in ok_rows]

    y_true = mlb.transform(y_true_labels)
    y_pred = mlb.transform(y_pred_labels)

    return {
        "path": path,
        "n_total_rows": len(all_rows),
        "n_examples": len(ok_rows),
        "n_error_rows": len(error_rows),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "avg_predicted_labels": np.mean([len(row["predicted_labels"]) for row in ok_rows]),
        "avg_gold_labels": np.mean([len(row["gold_labels"]) for row in ok_rows]),
        "avg_input_tokens": np.mean([row["input_tokens"] for row in ok_rows]),
        "avg_output_tokens": np.mean([row["output_tokens"] for row in ok_rows]),
        "avg_runtime_seconds": np.mean([row["wall_time_seconds"] for row in ok_rows])
    }

##Evaluation of the validation set results on 10 results (for testing/adjusting the prompt)

In [ ]:
# Saved check output files
GREEDY_CHECK_OUTPUT = f"{OUTPUT_DIR}/CHECK_llama_OLLAMA_val_greedy_predictions.jsonl"
TOPK_CHECK_OUTPUT = f"{OUTPUT_DIR}/CHECK_llama_val_OLLAMA_topk_predictions.jsonl"
LABELS_ONLY_CHECK_OUTPUT = f"{OUTPUT_DIR}/CHECK_llama_OLLAMA_val_labels_only_predictions.jsonl"

In [ ]:
run_ollama_validation_checkpointed(
    items=val_greedy_items,
    output_path=GREEDY_CHECK_OUTPUT,
    strategy_name="CHECK_greedy_5_examples_label_descriptions",
    build_messages_fn=build_greedy_messages,
    gold_labels_dict=val_gold_labels,
    max_items=10)

Already completed: 0
Remaining to run: 10
Finished. Total wall time: 0.43 minutes


In [ ]:
run_ollama_validation_checkpointed(
    items=val_topk_items,
    output_path=TOPK_CHECK_OUTPUT,
    strategy_name="CHECK_topk_5_examples_label_descriptions",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=val_gold_labels,
    max_items=10)

Already completed: 0
Remaining to run: 10
Finished. Total wall time: 0.62 minutes


In [ ]:
run_ollama_validation_checkpointed(
    items=val_greedy_items,
    output_path=LABELS_ONLY_CHECK_OUTPUT,
    strategy_name="CHECK_labels_only",
    build_messages_fn=build_labels_only_messages,
    gold_labels_dict=val_gold_labels,
    max_items=10)

Already completed: 0
Remaining to run: 10
Finished. Total wall time: 0.28 minutes


In [ ]:
check_results = pd.DataFrame([
    evaluate_prediction_file(GREEDY_CHECK_OUTPUT),
    evaluate_prediction_file(TOPK_CHECK_OUTPUT),
    evaluate_prediction_file(LABELS_ONLY_CHECK_OUTPUT)])

check_results

check_results_path = f"{OUTPUT_DIR}/CHECK_AAPD_llama_validation_strategy_results_comparison.csv"
check_results.to_csv(check_results_path, index=False)
#macrof1 is low  here because it uses the full label set (this will work fine in the final full run), but everything else works correctly

In [ ]:
check_results

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/AAPD_LLA...,10,10,0,0.434783,0.091975,2.3,2.3,3399.0,19.3,2.582248
1,/content/drive/MyDrive/thesis_results/AAPD_LLA...,10,10,0,0.545455,0.115638,2.1,2.3,3374.2,17.5,3.696423
2,/content/drive/MyDrive/thesis_results/AAPD_LLA...,10,10,0,0.250000,0.058642,2.5,2.3,2338.3,25.2,1.687758


In [ ]:
rows = []

with open(GREEDY_CHECK_OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df_greedy = pd.DataFrame(rows)

df_greedy.head(10)

,status,strategy,query_id,gold_labels,predicted_labels,raw_output,input_tokens,output_tokens,generation_runtime_seconds,total_sample_runtime_seconds,wall_time_seconds,timestamp
0,ok,CHECK_greedy_5_examples_label_descriptions,0,"[Digital Libraries, Information Retrieval]","[Databases, Computational Engineering, Finance...","{""predicted_labels"": [""Databases"", ""Computatio...",3267,24,0.727959,4.226428,4.228189,2026-06-06T22:21:04.416526
1,ok,CHECK_greedy_5_examples_label_descriptions,1,"[Information Theory (Computer Science), Inform...","[Computational Geometry, Numerical Analysis (C...","{""predicted_labels"": [""Computational Geometry""...",2831,19,0.535225,1.619256,1.619992,2026-06-06T22:21:06.056690
2,ok,CHECK_greedy_5_examples_label_descriptions,2,"[Information Theory (Computer Science), Inform...","[Information Theory (Computer Science), Comput...","{""predicted_labels"": [""Information Theory (Com...",3450,17,0.509626,2.212587,2.213655,2026-06-06T22:21:08.273916
3,ok,CHECK_greedy_5_examples_label_descriptions,3,"[Computation and Language, Information Retriev...","[Computational Linguistics, Machine Learning (...","{""predicted_labels"": [""Computational Linguisti...",3592,19,0.587403,2.789365,2.790992,2026-06-06T22:21:11.069197
4,ok,CHECK_greedy_5_examples_label_descriptions,4,"[Information Theory (Computer Science), Discre...","[Computational Complexity, Information Theory ...","{""predicted_labels"": [""Computational Complexit...",3487,18,0.564792,2.635711,2.636805,2026-06-06T22:21:13.712540
5,ok,CHECK_greedy_5_examples_label_descriptions,5,"[Information Theory (Computer Science), Disord...","[Information Theory (Computer Science), Statis...","{""predicted_labels"": [""Information Theory (Com...",3564,22,0.708815,2.563260,2.564287,2026-06-06T22:21:16.280520
6,ok,CHECK_greedy_5_examples_label_descriptions,6,"[Information Theory (Computer Science), Inform...","[Computational Complexity, Numerical Analysis ...","{""predicted_labels"": [""Computational Complexit...",3435,21,0.685405,2.414930,2.416170,2026-06-06T22:21:18.700639
7,ok,CHECK_greedy_5_examples_label_descriptions,7,"[Logic in Computer Science, Computational Comp...","[Logic in Computer Science, Computational Comp...","{""predicted_labels"": [""Logic in Computer Scien...",3341,16,0.520226,2.155343,2.156289,2026-06-06T22:21:20.861663
8,ok,CHECK_greedy_5_examples_label_descriptions,8,"[Information Theory (Computer Science), Inform...","[Information Theory (Computer Science), Networ...","{""predicted_labels"": [""Information Theory (Com...",3365,18,0.590725,2.257981,2.259171,2026-06-06T22:21:23.124761
9,ok,CHECK_greedy_5_examples_label_descriptions,9,"[Formal Languages and Automata Theory, Computa...","[Computational Complexity, Formal Languages an...","{""predicted_labels"": [""Computational Complexit...",3658,19,0.645566,2.935433,2.936927,2026-06-06T22:21:26.069233


In [ ]:
rows = []

with open(TOPK_CHECK_OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df_topk = pd.DataFrame(rows)

df_topk.head(10)

,status,strategy,query_id,gold_labels,predicted_labels,raw_output,input_tokens,output_tokens,generation_runtime_seconds,total_sample_runtime_seconds,wall_time_seconds,timestamp
0,ok,CHECK_topk_5_examples_label_descriptions,0,"[Digital Libraries, Information Retrieval]","[Information Retrieval, Digital Libraries]","{""predicted_labels"": [""Information Retrieval"",...",3029,14,0.457656,2.072292,2.073472,2026-06-06T22:21:28.160187
1,ok,CHECK_topk_5_examples_label_descriptions,1,"[Information Theory (Computer Science), Inform...","[Computational Geometry, Data Analysis, Statis...","{""predicted_labels"": [""Computational Geometry""...",2926,18,0.613788,1.823040,1.824388,2026-06-06T22:21:29.993362
2,ok,CHECK_topk_5_examples_label_descriptions,2,"[Information Theory (Computer Science), Inform...","[Information Theory (Computer Science), Comput...","{""predicted_labels"": [""Information Theory (Com...",3611,17,0.602655,2.556783,2.558704,2026-06-06T22:21:32.555847
3,ok,CHECK_topk_5_examples_label_descriptions,3,"[Computation and Language, Information Retriev...","[Computation and Language, Information Retrieval]","{""predicted_labels"": [""Computation and Languag...",3609,16,0.554733,2.475575,2.476842,2026-06-06T22:21:35.036206
4,ok,CHECK_topk_5_examples_label_descriptions,4,"[Information Theory (Computer Science), Discre...","[Information Theory (Computer Science), Comput...","{""predicted_labels"": [""Information Theory (Com...",3454,17,0.587947,2.396240,2.397633,2026-06-06T22:21:37.437725
5,ok,CHECK_topk_5_examples_label_descriptions,5,"[Information Theory (Computer Science), Disord...","[Information Theory (Computer Science), Statis...","{""predicted_labels"": [""Information Theory (Com...",3377,22,0.807731,2.878181,2.880920,2026-06-06T22:21:40.323892
6,ok,CHECK_topk_5_examples_label_descriptions,6,"[Information Theory (Computer Science), Inform...","[Computational Complexity, Numerical Analysis ...","{""predicted_labels"": [""Computational Complexit...",3416,19,0.692653,2.728302,2.729450,2026-06-06T22:21:43.060106
7,ok,CHECK_topk_5_examples_label_descriptions,7,"[Logic in Computer Science, Computational Comp...","[Logic in Computer Science, Computational Comp...","{""predicted_labels"": [""Logic in Computer Scien...",3165,16,0.579176,14.831339,14.832422,2026-06-06T22:21:57.896716
8,ok,CHECK_topk_5_examples_label_descriptions,8,"[Information Theory (Computer Science), Inform...","[Information Theory (Computer Science), Comput...","{""predicted_labels"": [""Information Theory (Com...",3360,17,0.617365,2.314505,2.315800,2026-06-06T22:22:00.216526
9,ok,CHECK_topk_5_examples_label_descriptions,9,"[Formal Languages and Automata Theory, Computa...","[Computational Complexity, Formal Languages an...","{""predicted_labels"": [""Computational Complexit...",3795,19,0.711118,2.873504,2.874600,2026-06-06T22:22:03.102623


In [ ]:
rows = []

with open(LABELS_ONLY_CHECK_OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df_labels = pd.DataFrame(rows)

df_labels.head(10)

,status,strategy,query_id,gold_labels,predicted_labels,raw_output,input_tokens,output_tokens,generation_runtime_seconds,total_sample_runtime_seconds,wall_time_seconds,timestamp
0,ok,CHECK_labels_only,0,"[Digital Libraries, Information Retrieval]","[Applications, Information Retrieval]","{\n ""predicted_labels"": [""Applications"", ""Inf...",2316,16,0.576665,1.214809,1.215139,2026-06-06T22:22:04.331464
1,ok,CHECK_labels_only,1,"[Information Theory (Computer Science), Inform...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2241,24,0.897293,1.425381,1.425591,2026-06-06T22:22:05.764340
2,ok,CHECK_labels_only,2,"[Information Theory (Computer Science), Inform...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2315,30,1.160526,1.776997,1.777319,2026-06-06T22:22:07.545255
3,ok,CHECK_labels_only,3,"[Computation and Language, Information Retriev...","[Machine Learning (Computer Science), Computat...","{\n ""predicted_labels"": [""Machine Learning (C...",2360,26,0.992913,1.995241,1.995652,2026-06-06T22:22:09.545630
4,ok,CHECK_labels_only,4,"[Information Theory (Computer Science), Discre...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2320,24,0.907880,1.894184,1.894540,2026-06-06T22:22:11.445086
5,ok,CHECK_labels_only,5,"[Information Theory (Computer Science), Disord...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2337,34,1.325840,1.956625,1.957024,2026-06-06T22:22:13.405641
6,ok,CHECK_labels_only,6,"[Information Theory (Computer Science), Inform...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2346,32,1.272879,1.914710,1.915085,2026-06-06T22:22:15.324645
7,ok,CHECK_labels_only,7,"[Logic in Computer Science, Computational Comp...","[Computational Complexity, Logic in Computer S...","{\n ""predicted_labels"": [""Computational Compl...",2292,19,0.718994,1.329982,1.330269,2026-06-06T22:22:16.658819
8,ok,CHECK_labels_only,8,"[Information Theory (Computer Science), Inform...","[Adaptation and Self-Organizing Systems, Compu...","{\n ""predicted_labels"": [""Adaptation and Self...",2403,29,1.193275,1.895165,1.895604,2026-06-06T22:22:18.558138
9,ok,CHECK_labels_only,9,"[Formal Languages and Automata Theory, Computa...","[Computational Complexity, Quantum Physics]","{\n ""predicted_labels"": [""Computational Compl...",2453,18,0.723724,1.470958,1.471361,2026-06-06T22:22:20.033512


##Full validation run

In [ ]:
run_ollama_validation_checkpointed(
    items=val_greedy_items,
    output_path=GREEDY_VAL_OUTPUT,
    strategy_name="greedy_5_examples_label_descriptions",
    build_messages_fn=build_greedy_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 1000
Remaining to run: 0
Finished. Total wall time: 0.00 minutes


In [ ]:
run_ollama_validation_checkpointed(
    items=val_topk_items,
    output_path=TOPK_VAL_OUTPUT,
    strategy_name="topk_5_examples_label_descriptions",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 0
Remaining to run: 1000
Finished. Total wall time: 45.62 minutes


In [ ]:
run_ollama_validation_checkpointed(
    items=val_greedy_items,
    output_path=LABELS_ONLY_VAL_OUTPUT,
    strategy_name="labels_only",
    build_messages_fn=build_labels_only_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 0
Remaining to run: 1000
Finished. Total wall time: 31.30 minutes


In [ ]:
results = pd.DataFrame([
    evaluate_prediction_file(GREEDY_VAL_OUTPUT),
    evaluate_prediction_file(TOPK_VAL_OUTPUT),
    evaluate_prediction_file(LABELS_ONLY_VAL_OUTPUT)])

results_path = f"{OUTPUT_DIR}/AAPD_llama_validation_strategy_results_comparison.csv"
results.to_csv(results_path, index=False)

results
#topk + label descriptions performs best

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.555986,0.484232,2.244,2.4,3483.481,19.111,2.789963
1,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.589140,0.516064,2.149,2.4,3454.613,18.503,2.731964
2,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.264241,0.247169,2.656,2.4,2372.683,27.517,1.872958


#**Final run on test set**

In [ ]:
TEST_TOPK_INPUT = "aapd_test_top_k_llm_input_data.jsonl"
TOPK_TEST_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_test_topk_predictions.jsonl"
TEST_GOLD_LABELS = "aapd_test_gold_labels.json"

In [ ]:
test_topk_items = load_jsonl(TEST_TOPK_INPUT)

with open(TEST_GOLD_LABELS, "r", encoding="utf-8") as f:
    test_gold_labels = json.load(f)

print("Test top-k items:", len(test_topk_items))
print("Test gold labels:", len(test_gold_labels))

Test top-k items: 1000
Test gold labels: 1000


In [ ]:
run_ollama_validation_checkpointed(
    items=test_topk_items,
    output_path=TOPK_TEST_OUTPUT,
    strategy_name="topk_5_examples_label_descriptions_ollama_schema",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=test_gold_labels,
    max_items=None)

Already completed: 0
Remaining to run: 1000
Finished. Total wall time: 46.88 minutes


In [ ]:
test_results = evaluate_prediction_file(TOPK_TEST_OUTPUT)
pd.DataFrame([test_results])

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.581096,0.506392,2.16,2.421,3461.618,18.707,2.80799


In [ ]:
all_rows = load_prediction_jsonl_all(TOPK_TEST_OUTPUT)

ok_rows = [row for row in all_rows if row.get("status") == "ok"]
error_rows = [row for row in all_rows if row.get("status") == "error"]

if len(error_rows) > 0:
    raise ValueError(f"{TOPK_TEST_OUTPUT} contains {len(error_rows)} error rows.")

llm_y_test_true_labels = [row["gold_labels"] for row in ok_rows]
llm_y_test_pred_labels = [row["predicted_labels"] for row in ok_rows]

llm_y_test = mlb.transform(llm_y_test_true_labels)
llm_y_test_pred = mlb.transform(llm_y_test_pred_labels)

report_dict = classification_report(llm_y_test, llm_y_test_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(f"{OUTPUT_DIR}/AAPD_Llama_Ollama_topk_classification_report.csv")

report_df


,precision,recall,f1-score,support
Adaptation and Self-Organizing Systems,0.133333,0.250000,0.173913,8.0
Applications,0.071429,0.071429,0.071429,14.0
Artificial Intelligence,0.591837,0.522523,0.555024,111.0
Combinatorics,0.727273,0.533333,0.615385,60.0
Computation and Language,0.793103,0.901961,0.844037,51.0
Computational Complexity,0.194286,0.772727,0.310502,44.0
"Computational Engineering, Finance, and Science",0.154930,0.458333,0.231579,24.0
Computational Geometry,0.592593,0.727273,0.653061,22.0
Computational Linguistics,0.846154,0.733333,0.785714,15.0
Computer Science and Game Theory,0.709677,0.758621,0.733333,29.0


In [ ]:
result = {
    "model": "Llama3.1_8B_Ollama_q4_K_M",
    "dataset": "AAPD",
    "prompt_strategy": "topk_5_examples_label_descriptions",
    "test_f1_micro": f1_score(llm_y_test, llm_y_test_pred, average="micro", zero_division=0),
    "test_f1_macro": f1_score(llm_y_test, llm_y_test_pred, average="macro", zero_division=0),
    "avg_predicted_labels": np.mean([len(row["predicted_labels"]) for row in ok_rows]),
    "avg_inference_time_sec": np.mean([row["total_sample_runtime_seconds"] for row in ok_rows]),
    "inference_per_sample_ms": np.mean([row["total_sample_runtime_seconds"] for row in ok_rows]) * 1000,
    "total_wall_time_min": 46.88,
    "n_error_rows": len(error_rows)}

result_df = pd.DataFrame([result])
result_df.to_csv(f"{OUTPUT_DIR}/AAPD_Llama_Ollama_topk_test_results_summary.csv", index=False)
result

{'model': 'Llama3.1_8B_Ollama_q4_K_M',
 'dataset': 'AAPD',
 'prompt_strategy': 'topk_5_examples_label_descriptions',
 'test_f1_micro': 0.5810958306046715,
 'test_f1_macro': 0.5063924109013279,
 'avg_predicted_labels': np.float64(2.16),
 'avg_inference_time_sec': np.float64(2.806423471927643),
 'inference_per_sample_ms': np.float64(2806.423471927643),
 'total_wall_time_min': 46.88,
 'n_error_rows': 0}